In [1]:
import mne
import pandas as pd
import scipy
import numpy as np

In [2]:
def import_sorted_fif_files():
    fif_files = ["P1_T1.fif","P1_T2.fif","P1_T3.fif"]
    return [mne.io.read_raw_fif(f, preload=True) for f in fif_files]

In [3]:
fif_task_1 = import_sorted_fif_files()

Opening raw data file P1_T1.fif...
    Range : 0 ... 24831 =      0.000 ...    96.996 secs
Ready.
Reading 0 ... 24831  =      0.000 ...    96.996 secs...
Opening raw data file P1_T2.fif...
    Range : 0 ... 86527 =      0.000 ...   337.996 secs
Ready.
Reading 0 ... 86527  =      0.000 ...   337.996 secs...
Opening raw data file P1_T3.fif...
    Range : 0 ... 168447 =      0.000 ...   657.996 secs
Ready.
Reading 0 ... 168447  =      0.000 ...   657.996 secs...


C:\Users\Suyash\AppData\Local\Temp\ipykernel_8404\10331175.py:3: RuntimeWarning: This filename (P1_T1.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  return [mne.io.read_raw_fif(f, preload=True) for f in fif_files]
C:\Users\Suyash\AppData\Local\Temp\ipykernel_8404\10331175.py:3: RuntimeWarning: This filename (P1_T2.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  return [mne.io.read_raw_fif(f, preload=True) for f in fif_files]
C:\Users\Suyash\AppData\Local\Temp\ipykernel_8404\10331175.py:3: RuntimeWarning: This filename (P1_T3.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.

In [4]:
fif_1_baseline = []
for fif in fif_task_1 :
    fif_temp = fif.copy()
    t1 = 0
    t2 = 30
    fif_task_baseline = fif_temp.crop(tmin=t1, tmax=t2)
    fif_1_baseline.append(fif_task_baseline)

In [5]:
fif_1_sketching_signals = []
for fif in fif_task_1 :
    fif_temp = fif.copy()
    t1 = 30
    t2 = fif_temp.times[-1]
    fif_task_signal = fif_temp.crop(tmin=t1, tmax=t2)
    fif_1_sketching_signals.append(fif_task_signal)

In [6]:
def epoch_equally(rawf_8, epoch_duration):
    # Create fixed-length events
    event_id = 1  # Assign a label to the events
    event_duration = epoch_duration
    events = mne.make_fixed_length_events(rawf_8, id=event_id, duration=event_duration)

    # Define epoch parameters
    tmin = 0  # Start at the event
    tmax = epoch_duration # End epoch_duration seconds after the event

    # Create epochs
    epochs_8 = mne.Epochs(rawf_8, events, event_id=event_id, tmin=tmin, tmax=tmax, reject_by_annotation= True, baseline=None, preload=True)
    return epochs_8

In [7]:
def concatenate_epochs_into_channels(epochs_8):

    epochs_8_data = epochs_8.get_data()
    n_epochs, n_channels, n_samples = epochs_8_data.shape

    # Initialize a list to store concatenated arrays for each channel
    concatenated_channels = []

    # Loop through each channel
    for channel in range(n_channels):
        # Extract the first row of the current channel across all epochs
        concatenated = np.concatenate([epochs_8_data[epoch, channel, :] for epoch in range(n_epochs)])
        concatenated_channels.append(concatenated)

    # Convert to a NumPy array if needed
    concatenated_channels = np.array(concatenated_channels)

    af3_signal_array = concatenated_channels[0][:]
    f7_signal_array = concatenated_channels[1][:]
    f3_signal_array = concatenated_channels[2][:]
    fc5_signal_array = concatenated_channels[3][:]
    fc6_signal_array = concatenated_channels[4][:]
    f4_signal_array = concatenated_channels[5][:]
    f8_signal_array = concatenated_channels[6][:]
    af4_signal_array = concatenated_channels[7][:]

    sketch_signal_length = len(af3_signal_array)/256


    eeg_channels_data = {
        'AF3': af3_signal_array ,
        'F7': f7_signal_array ,
        'F3': f3_signal_array ,
        'FC5': fc5_signal_array ,
        'FC6': fc6_signal_array,
        'F4': f4_signal_array,
        'F8': f8_signal_array,
        'AF4': af4_signal_array,
    }

    return eeg_channels_data, sketch_signal_length


In [8]:
def channelwise_abp(eeg_channels_data_dictionary):
    sf = 256
    bands = {
        "delta": [1, 4],
        "theta": [4, 8],
        "alpha": [8, 13],
        "beta": [13, 30],
        "brain": [1, 30],
        "full": [0, 128]
    }

    # Extract channel signals
    channels = ["AF3", "F7", "F3", "FC5", "FC6", "F4", "F8", "AF4"]
    all_abp_set = []

    for channel in channels:
        signal = eeg_channels_data_dictionary[channel]
        abp_set = []
        for band_name, band_range in bands.items():
            abp, _ = bandpower(signal, sf, band_range, method='multitaper', window_sec=None, relative=False)
            abp_set.append(abp)
        all_abp_set.append(abp_set)

    # Convert to NumPy array
    all_abp_set = np.array(all_abp_set)
    return all_abp_set


In [9]:
def bandpower(data, sf, band, method='welch', window_sec=None, relative=False):
    """Compute the average power of the signal x in a specific frequency band.
    Requires MNE-Python >= 0.14.

    Parameters
    ----------
    data : 1d-array
      Input signal in the time-domain.
    sf : float
      Sampling frequency of the data.
    band : list
      Lower and upper frequencies of the band of interest.
    method : string
      Periodogram method: 'welch' or 'multitaper'
    window_sec : float
      Length of each window in seconds. Useful only if method == 'welch'.
      If None, window_sec = (1 / min(band)) * 2.
    relative : boolean
      If True, return the relative power (= divided by the total power of the signal).
      If False (default), return the absolute power.

    Return
    ------
    bp : float
      Absolute or relative band power.
    freq_res : float
      Frequency resolution of the bandpower calculation
    """
    from scipy.signal import welch
    from scipy.integrate import simpson
    from mne.time_frequency import psd_array_multitaper

    band = np.asarray(band)
    low, high = band

    # Compute the modified periodogram (Welch)
    if method == 'welch':
        if window_sec is not None:
            nperseg = window_sec * sf
        else:
            nperseg = (2 / low) * sf

        freqs, psd = welch(data, sf, nperseg=nperseg)

    elif method == 'multitaper':
        psd, freqs = psd_array_multitaper(data, sf, adaptive=True,
                                          normalization='full', n_jobs=1, remove_dc= True, max_iter = 1024, verbose=0)

    # Frequency resolution
    freq_res = freqs[1] - freqs[0]

    # Find index of band in frequency vector
    idx_band = np.logical_and(freqs >= low, freqs <= high)

    # Integral approximation of the spectrum using parabola (Simpson's rule)
    bp = simpson(psd[idx_band], dx=freq_res)

    if relative:
        bp /= simpson(psd, dx=freq_res)
    return bp, freq_res

In [12]:
epoch_duration_abp = 5
fif_1_participantwise_abp = []
fif_1_bl_participantwise_abp = []
fif_1_participantwise_abp_df = []
fif_1_bl_participantwise_abp_df = []
fif_1_signal_length_baseline = []
fif_1_signal_length_sketch = []

temp_epoc = None
for idx,fif in enumerate(fif_1_baseline):
    print("----------------------------")
    bl_sketch_temp = fif
    bl_sketch_temp_epochs = epoch_equally(bl_sketch_temp,epoch_duration_abp)
    bl_channelwise_data_temp, signal_length_baseline_temp = concatenate_epochs_into_channels(bl_sketch_temp_epochs)
    bl_abp_list_temp = channelwise_abp(bl_channelwise_data_temp)
    print(bl_abp_list_temp)
    break
    # fif_1_bl_participantwise_abp.append(bl_abp_list_temp)
    # fif_1_signal_length_baseline.append(signal_length_baseline_temp)

    # channel_names = np.array(["AF3", "F7", "F3", "FC5", "FC6", "F4", "F8", "AF4"])
    # bl_abp_df_list_temp = pd.DataFrame(bl_abp_list_temp, columns=['bl_delta_abp','bl_theta_abp','bl_alpha_abp','bl_beta_abp','bl_brain_abp','bl_full_abp'])
    # pd.options.display.float_format ='{:.2e}'.format
    # bl_abp_df_list_temp['Channel'] = channel_names
    # fif_1_bl_participantwise_abp_df.append(bl_abp_df_list_temp)

----------------------------
Not setting metadata
6 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 6 events and 1281 original time points ...
0 bad epochs dropped
[[3.76820923e-11 1.70873130e-11 6.30589803e-11 2.07980993e-11
  1.39091995e-10 1.52613927e-10]
 [1.39517062e-11 6.44393122e-12 2.34212509e-11 7.56064960e-12
  5.15503149e-11 5.64460182e-11]
 [2.14284637e-11 1.12101364e-11 3.38834493e-11 1.07385456e-11
  7.75320507e-11 8.56858529e-11]
 [2.49423067e-11 1.04745515e-11 4.01678450e-11 1.50092706e-11
  9.08995839e-11 1.01310747e-10]
 [1.23671500e-11 5.63859139e-12 2.07201944e-11 6.78822187e-12
  4.56670445e-11 5.00727059e-11]
 [4.85397995e-11 2.14348972e-11 8.05297585e-11 2.75563940e-11
  1.78658574e-10 1.96836645e-10]
 [1.64546147e-11 7.55775042e-12 2.76034906e-11 8.96561197e-12
  6.07850851e-11 6.65943600e-11]
 [2.91892373e-11 1.32363373e-11 4.88468057e-11 1.61103477e-11
  1.07743321e-10 1.18217446e-10]]


In [11]:
fif_1_participantwise_abp

[]